# 像素坐标与深度到相机坐标

这一节是正向投影的逆过程：给定像素坐标和该像素处的深度，恢复相机坐标系中的三维点。需要时，再使用外参恢复到世界坐标系。

反投影是生成彩色点云、$6\mathrm{DoF}$ 位姿估计和三维重建的基础。

## 1. 坐标约定

通常以相机光心作为原点：

$$
O_c=(0,0,0)
$$

常见的 OpenCV 相机坐标系约定为：$X_c$ 向右、$Y_c$ 向下、$Z_c$ 向前。像素坐标系原点通常位于图像左上角，$u$ 轴向右，$v$ 轴向下。


## 2. 像素坐标与深度 → 相机坐标

已知像素坐标 $(u,v)$ 和深度图在该像素处的值 $Z_c=D(u,v)$，可以恢复相机坐标系中的三维点。这里的深度必须表示点的相机坐标 $Z_c$，而不是相机光心到点的直线距离。

核心公式为：

$$
\mathbf{P}_c=Z_c\mathbf{K}^{-1}\begin{bmatrix}u\\v\\1\end{bmatrix}
=\begin{bmatrix}
\dfrac{(u-c_x)Z_c}{f_x}\\
\dfrac{(v-c_y)Z_c}{f_y}\\
Z_c
\end{bmatrix}
$$

其中相机内参矩阵为：

$$
\mathbf{K}=\begin{bmatrix}
f_x&0&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix}
$$


## 3. 矩阵形式

像素齐次坐标为：

$$
\tilde{\mathbf{p}}=\begin{bmatrix}u\\v\\1\end{bmatrix}
$$

内参矩阵的逆为：

$$
\mathbf{K}^{-1}=\begin{bmatrix}
\dfrac{1}{f_x}&0&-\dfrac{c_x}{f_x}\\
0&\dfrac{1}{f_y}&-\dfrac{c_y}{f_y}\\
0&0&1
\end{bmatrix}
$$

因此：

$$
\boxed{\mathbf{P}_c=Z_c\mathbf{K}^{-1}\tilde{\mathbf{p}}}
$$


## 4. 恢复到世界坐标系

若采用世界坐标系到相机坐标系的外参约定，并满足：

$$
\mathbf{P}_c=\mathbf{R}\mathbf{P}_w+\mathbf{t}
$$

则反变换为：

$$
\mathbf{P}_w=\mathbf{R}^{T}(\mathbf{P}_c-\mathbf{t})
$$


## 5. 有效深度与注意事项

通常只对满足以下条件的深度进行反投影：

$$
D(u,v)>0\quad\text{且}\quad D(u,v)\text{ 为有限值}
$$

还需要注意：

- 反投影结果的单位与深度值相同；毫米深度会得到毫米坐标。
- 原始图像存在镜头畸变时，应先使用去畸变后的像素坐标，或显式加入畸变模型。
- 不同系统的坐标轴约定可能不同，必须确认 $X_c$、$Y_c$、$Z_c$ 的方向。


## 6. 验收实验

- 给定至少 5 组像素坐标和深度，恢复相机坐标系中的三维点。
- 将恢复结果与已知三维点比较，计算位置误差。
- 对一组已知的世界坐标点先进行正向投影，再使用对应深度执行反投影，完成闭环验证：

$$
\mathbf{P}_w\rightarrow\mathbf{P}_c\rightarrow(u,v)\rightarrow\mathbf{P}'_c
$$

- 解释误差可能来自浮点计算、像素取整、深度噪声或内外参不一致。

In [ ]:
import numpy as np

def pixel_to_camera(u, v, depth, K):
    """将像素坐标和深度反投影到相机坐标系下的 3D 点。"""
    if not np.isfinite(depth) or depth <= 0:
        raise ValueError("depth 必须是正的有限值")

    pixel_h = np.array([u, v, 1.0], dtype=float)
    return depth * (np.linalg.solve(K, pixel_h))

In [ ]:
K = np.array([
    [1000.0, 0.0, 960.0],
    [0.0, 1000.0, 540.0],
    [0.0, 0.0, 1.0]
])

# 像素坐标、深度和预期相机坐标
tests = [
    ((960.0, 540.0), 5.0, np.array([0.0, 0.0, 5.0])),
    ((1160.0, 640.0), 5.0, np.array([1.0, 0.5, 5.0])),
    ((760.0, 640.0), 5.0, np.array([-1.0, 0.5, 5.0])),
    ((1160.0, 440.0), 5.0, np.array([1.0, -0.5, 5.0])),
    ((1060.0, 590.0), 10.0, np.array([1.0, 0.5, 10.0])),
]

for i, ((u, v), depth, expected) in enumerate(tests, start=1):
    recovered = pixel_to_camera(u, v, depth, K)
    error = np.linalg.norm(recovered - expected)
    print(f"测试点 {i}: 恢复坐标={recovered}, 误差={error:.10f}")